In [22]:
import pandas as pd
import numpy as np

In [23]:

# --------------------------------------------------
# 1. GINI IMPURITY
# --------------------------------------------------

def gini(y):
    classes, counts = np.unique(y, return_counts=True)

    probabilities = counts / len(y)

    return 1 - np.sum(probabilities ** 2)

In [24]:
# --------------------------------------------------
# 2. SPLIT DATA
# --------------------------------------------------

def split_data(X, y, feature, threshold):

    left = X[:, feature] <= threshold
    right = X[:, feature] > threshold

    X_left = X[left]
    y_left = y[left]

    X_right = X[right]
    y_right = y[right]

    return X_left, y_left, X_right, y_right

In [25]:
# --------------------------------------------------
# 3. INFORMATION GAIN
# --------------------------------------------------

def information_gain(y, y_left, y_right):

    parent_gini = gini(y)

    n = len(y)
    n_left = len(y_left)
    n_right = len(y_right)

    weighted_gini = (
        (n_left / n) * gini(y_left)
        + (n_right / n) * gini(y_right)
    )

    return parent_gini - weighted_gini

In [26]:
# --------------------------------------------------
# 4. FIND BEST SPLIT
# --------------------------------------------------

def best_split(X, y):

    best_gain = -1
    best_feature = None
    best_threshold = None

    n_features = X.shape[1]

    for feature in range(n_features):

        values = np.unique(X[:, feature])

        for threshold in values:

            X_left, y_left, X_right, y_right = \
                split_data(X, y, feature, threshold)

            # Ignore empty splits
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            gain = information_gain(
                y,
                y_left,
                y_right
            )

            if gain > best_gain:

                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gain


In [27]:
# --------------------------------------------------
# 5. CREATE TREE
# --------------------------------------------------

def build_tree(X, y, depth=0, max_depth=3):

    # If all values belong to same class
    if len(np.unique(y)) == 1:

        return int(y[0])

    # Stop if maximum depth reached
    if depth == max_depth:

        values, counts = np.unique(
            y,
            return_counts=True
        )

        return int(values[np.argmax(counts)])

    # Find best split
    feature, threshold, gain = best_split(X, y)

    # No useful split
    if feature is None:

        values, counts = np.unique(
            y,
            return_counts=True
        )

        return int(values[np.argmax(counts)])

    # Split data
    X_left, y_left, X_right, y_right = \
        split_data(X, y, feature, threshold)

    # Recursively create branches
    left_tree = build_tree(
        X_left,
        y_left,
        depth + 1,
        max_depth
    )

    right_tree = build_tree(
        X_right,
        y_right,
        depth + 1,
        max_depth
    )

    return {
        "feature": feature,
        "threshold": threshold,
        "left": left_tree,
        "right": right_tree
    }

In [28]:
# --------------------------------------------------
# 6. PREDICT ONE SAMPLE
# --------------------------------------------------

def predict_one(tree, sample):

    # Leaf node
    if not isinstance(tree, dict):

        return tree

    feature = tree["feature"]
    threshold = tree["threshold"]

    if sample[feature] <= threshold:

        return predict_one(
            tree["left"],
            sample
        )

    else:

        return predict_one(
            tree["right"],
            sample
        )


In [29]:
# --------------------------------------------------
# 7. PREDICT MULTIPLE SAMPLES
# --------------------------------------------------

def predict(tree, X):

    predictions = []

    for sample in X:

        predictions.append(
            predict_one(tree, sample)
        )

    return np.array(predictions)

In [30]:
# ==================================================
# DATASET
# ==================================================

# Features:
# 0 = Income
# 1 = Credit Score

X = np.array([
    [20, 500],
    [25, 550],
    [30, 600],
    [40, 650],
    [50, 700],
    [60, 750]
])

# 0 = No
# 1 = Yes

y = np.array([
    0,
    0,
    1,
    1,
    1,
    1
])

In [31]:
# ==================================================
# TRAIN
# ==================================================

tree = build_tree(
    X,
    y,
    max_depth=3
)


print("Decision Tree:")
print(tree)

Decision Tree:
{'feature': 0, 'threshold': np.int64(25), 'left': 0, 'right': 1}


In [32]:
# ==================================================
# PREDICTION
# ==================================================

X_test = np.array([
    [27, 570],
    [45, 680],
    [22, 520]
])

predictions = predict(tree, X_test)

print("\nPredictions:")
print(predictions)


Predictions:
[1 1 0]
